# Phase 8 : Chargement et Documentation

## Objectif

Cette phase prépare les livrables finaux du projet afin d'assurer leur
reproductibilité, leur vérification et leur partage.

Les principaux livrables sont :

- le dataset supervisé final ;
- la pipeline Machine Learning entraînée ;
- les métadonnées du modèle ;
- le Data Dictionary ;
- la documentation des transformations ;
- les informations nécessaires pour reproduire le projet.

In [25]:
from pathlib import Path
from datetime import date

import json
import joblib
import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd()


if (
    not (PROJECT_ROOT / "data").exists()
    and (PROJECT_ROOT / "ML_NBA_Project" / "data").exists()
):
    PROJECT_ROOT = (
        PROJECT_ROOT
        / "ML_NBA_Project"
    )


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

CURATED_DIR = (
    PROJECT_ROOT
    / "data"
    / "curated"
)

EXPORT_DIR = (
    PROJECT_ROOT
    / "exports"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)

DOC_DIR = (
    PROJECT_ROOT
    / "docs"
)


for directory in [
    CURATED_DIR,
    EXPORT_DIR,
    MODEL_DIR,
    DOC_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


VERSION = "v1"
DATE_TAG = date.today().strftime("%Y%m%d")


print(
    "Projet :",
    PROJECT_ROOT.resolve()
)

Projet : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model


## 1. Vérification du dataset final

Le dataset supervisé utilisé pendant la modélisation est rechargé avant son
export.

Il contient uniquement les joueur-saisons possédant une saison N+1 consécutive
permettant de construire la cible `pts_next_season`.

In [26]:
DATASET_PATH = (
    PROCESSED_DIR
    / "nba_player_season_supervised.parquet"
)


df = pd.read_parquet(
    DATASET_PATH
)


print(
    "Dimensions :",
    df.shape
)

print(
    "Période :",
    df["SEASON"].min(),
    "à",
    df["SEASON"].max()
)

print(
    "Cibles manquantes :",
    df["pts_next_season"].isna().sum()
)

print(
    "Doublons exacts :",
    df.duplicated().sum()
)

Dimensions : (9911, 251)
Période : 1996-97 à 2021-22
Cibles manquantes : 0
Doublons exacts : 0


## 2. Export du dataset final

Deux formats sont produits :

- **Parquet** pour préserver le typage et bénéficier de la compression ;
- **CSV UTF-8** pour faciliter le partage et la compatibilité avec d'autres outils.

Les fichiers suivent une convention de nommage contenant la version et la date
d'export.

In [27]:
PARQUET_EXPORT = (
    CURATED_DIR
    / f"nba_next_season_dataset_{VERSION}_{DATE_TAG}.parquet"
)

CSV_EXPORT = (
    EXPORT_DIR
    / f"nba_next_season_dataset_{VERSION}_{DATE_TAG}.csv"
)


df.to_parquet(
    PARQUET_EXPORT,
    index=False
)


df.to_csv(
    CSV_EXPORT,
    index=False,
    encoding="utf-8"
)


print(
    "Parquet :",
    PARQUET_EXPORT
)

print(
    "CSV :",
    CSV_EXPORT
)

Parquet : c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\curated\nba_next_season_dataset_v1_20261005.parquet
CSV : c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\exports\nba_next_season_dataset_v1_20261005.csv


In [28]:
parquet_check = pd.read_parquet(
    PARQUET_EXPORT
)


csv_check = pd.read_csv(
    CSV_EXPORT
)


print(
    "Original :",
    df.shape
)

print(
    "Parquet :",
    parquet_check.shape
)

print(
    "CSV :",
    csv_check.shape
)

Original : (9911, 251)
Parquet : (9911, 251)
CSV : (9911, 251)


In [29]:
assert (
    parquet_check.shape
    == df.shape
)

assert (
    csv_check.shape
    == df.shape
)

assert (
    parquet_check.columns.tolist()
    == df.columns.tolist()
)

assert (
    csv_check.columns.tolist()
    == df.columns.tolist()
)


print(
    "Exports vérifiés avec succès."
)

Exports vérifiés avec succès.


## 3. Vérification du modèle final

La pipeline Machine Learning complète est rechargée depuis le fichier `joblib`.

Cette vérification confirme que le preprocessing et le modèle Ridge peuvent être
réutilisés sans réexécuter le notebook de modélisation.

In [30]:
import sys

model_files = sorted(
    MODEL_DIR.glob(
        "nba_pts_ridge_pipeline_v1_*.joblib"
    )
)


if not model_files:
    raise FileNotFoundError(
        "Aucun modèle final trouvé dans models/"
    )


MODEL_PATH = model_files[-1]


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


from src.preprocessing import (
    normalize_categorical_missing
)


loaded_model = joblib.load(
    MODEL_PATH
)


print(
    "Fichier :",
    MODEL_PATH.name
)

print(
    "Modèle :",
    loaded_model.named_steps["modele"]
)

Fichier : nba_pts_ridge_pipeline_v1_20261005.joblib
Modèle : Ridge(alpha=100.0)


### Vérification de la pipeline chargée

La pipeline rechargée est utilisée sur quelques observations brutes du dataset
afin de vérifier qu'elle peut produire directement des prédictions.

In [31]:
excluded_columns = [
    "pts_next_season",
    "PLAYER_ID",
    "PLAYER_NAME",
    "NICKNAME",
    "TEAM_ID",
    "TEAM_ABBREVIATION",
    "SEASON",
]


X_demo = (
    df.drop(
        columns=excluded_columns
    )
    .iloc[:3]
)


demo_predictions = loaded_model.predict(
    X_demo
)


demo_results = pd.DataFrame({
    "PTS_N":
        X_demo["PTS"],

    "PTS_reel_N1":
        df.loc[
            X_demo.index,
            "pts_next_season"
        ],

    "PTS_predit_N1":
        demo_predictions,
})


display(
    demo_results
)

,PTS_N,PTS_reel_N1,PTS_predit_N1
0,5.0,3.5,4.299275
1,3.5,9.8,2.301156
2,9.8,4.8,7.930947


## 4. Métadonnées du modèle

Les principales informations nécessaires à la compréhension et à la
réutilisation du modèle sont exportées dans un fichier JSON.

In [32]:
model_metadata = {
    "project":
        "NBA Next Season Points ML",

    "version":
        VERSION,

    "creation_date":
        date.today().isoformat(),

    "target":
        "pts_next_season",

    "problem_type":
        "regression",

    "model":
        "Ridge",

    "alpha":
        100.0,

    "main_metric":
        "MAE",

    "train_period":
        "1996-97 to 2011-12",

    "validation_period":
        "2012-13 to 2016-17",

    "test_period":
        "2017-18 to 2021-22",

    "test_MAE":
        2.392930,

    "test_RMSE":
        3.104185,

    "test_R2":
        0.774507,

    "persistence_test_MAE":
        2.558404,

    "dataset_rows":
        int(df.shape[0]),

    "dataset_columns":
        int(df.shape[1]),

    "excluded_features":
        excluded_columns,

    "model_file":
        MODEL_PATH.name,
}

In [33]:
METADATA_PATH = (
    MODEL_DIR
    / f"nba_pts_model_metadata_{VERSION}_{DATE_TAG}.json"
)


with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


print(
    "Métadonnées sauvegardées :",
    METADATA_PATH
)

Métadonnées sauvegardées : c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\models\nba_pts_model_metadata_v1_20261005.json


In [34]:
with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:

    metadata_check = json.load(
        file
    )


display(
    pd.Series(
        metadata_check,
        name="valeur"
    )
)

project                                         NBA Next Season Points ML
version                                                                v1
creation_date                                                  2026-10-05
target                                                    pts_next_season
problem_type                                                   regression
model                                                               Ridge
alpha                                                               100.0
main_metric                                                           MAE
train_period                                           1996-97 to 2011-12
validation_period                                      2012-13 to 2016-17
test_period                                            2017-18 to 2021-22
test_MAE                                                          2.39293
test_RMSE                                                        3.104185
test_R2                               

## 5. Data Dictionary

Un dictionnaire de données est généré pour les 251 colonnes du dataset final.

Pour chaque variable, il contient :

- le type ;
- une description ;
- l'unité ou le type de valeur ;
- les valeurs possibles ou la plage observée ;
- la source ;
- la transformation appliquée ;
- le nombre et le pourcentage de valeurs manquantes.

L'objectif est de permettre à une personne extérieure au projet de comprendre
le contenu du dataset sans avoir à relire l'ensemble des notebooks.

In [35]:
derived_descriptions = {
    "pts_next_season":
        "Points par match du joueur lors de la saison N+1.",

    "PTS_PREV":
        "Points par match lors de la saison précédente.",

    "MIN_PREV":
        "Minutes par match lors de la saison précédente.",

    "AST_PREV":
        "Passes décisives par match lors de la saison précédente.",

    "REB_PREV":
        "Rebonds par match lors de la saison précédente.",

    "USG_PCT_PREV":
        "Taux d'utilisation lors de la saison précédente.",

    "PTS_CHANGE":
        "Variation des points par match par rapport à la saison précédente.",

    "MIN_CHANGE":
        "Variation des minutes par match par rapport à la saison précédente.",

    "AST_CHANGE":
        "Variation des passes décisives par rapport à la saison précédente.",

    "REB_CHANGE":
        "Variation des rebonds par rapport à la saison précédente.",

    "USG_PCT_CHANGE":
        "Variation du taux d'utilisation par rapport à la saison précédente.",

    "PTS_CAREER_AVG_PREV":
        "Moyenne historique des points avant la saison courante.",

    "MIN_CAREER_AVG_PREV":
        "Moyenne historique des minutes avant la saison courante.",

    "AST_CAREER_AVG_PREV":
        "Moyenne historique des passes avant la saison courante.",

    "REB_CAREER_AVG_PREV":
        "Moyenne historique des rebonds avant la saison courante.",

    "PTS_PER_MIN":
        "Nombre de points inscrits par minute jouée.",

    "FGA_PER_MIN":
        "Nombre de tirs tentés par minute jouée.",

    "FG3A_RATE":
        "Part des tirs tentés qui sont des tirs à trois points.",

    "FTA_RATE":
        "Rapport entre lancers francs tentés et tirs tentés.",

    "AST_TO_TOV":
        "Rapport entre passes décisives et pertes de balle.",

    "AGE_GROUP":
        "Classe d'âge du joueur.",

    "EXPERIENCE_GROUP":
        "Classe d'expérience NBA du joueur.",

    "HAS_PREVIOUS_SEASON":
        "Indique si une saison précédente est disponible.",

    "IS_ROOKIE":
        "Indique si le joueur est rookie.",

    "MULTI_TEAM_SEASON":
        "Indique si le joueur a évolué pour plusieurs équipes durant la saison.",

    "ROSTER_AVAILABLE":
        "Indique si les informations de roster sont disponibles.",
}

In [36]:
def get_source(column):
    if column in derived_descriptions:
        return "Feature engineering Phase 4"

    if column.startswith("TEAM_"):
        return "NBA API - statistiques équipe"

    if column in [
        "POSITION",
        "EXP_YEARS",
        "EXPERIENCE_GROUP",
        "ROSTER_AVAILABLE",
    ]:
        return "NBA API - roster"

    if column in [
        "AGE",
        "HEIGHT",
        "WEIGHT",
        "COUNTRY",
        "SCHOOL",
        "COLLEGE",
        "DRAFT_YEAR",
        "DRAFT_ROUND",
        "DRAFT_NUMBER",
    ]:
        return "NBA API - biographie joueur"

    return "NBA API - statistiques joueur"


def get_description(column):
    if column in derived_descriptions:
        return derived_descriptions[column]

    descriptions = {
        "PLAYER_ID": "Identifiant NBA unique du joueur.",
        "PLAYER_NAME": "Nom complet du joueur.",
        "NICKNAME": "Nom ou surnom du joueur.",
        "SEASON": "Saison NBA correspondant à l'observation.",
        "TEAM_ID": "Identifiant NBA de l'équipe.",
        "TEAM_ABBREVIATION": "Abréviation de l'équipe.",
        "AGE": "Âge du joueur lors de la saison.",
        "POSITION": "Poste indiqué dans le roster NBA.",
        "EXPERIENCE": "Nombre de saisons d'expérience NBA.",
    }

    if column in descriptions:
        return descriptions[column]

    if column.endswith("_PCT"):
        return f"Pourcentage ou ratio NBA correspondant à {column}."

    if column.endswith("_PG"):
        return f"Statistique NBA par match correspondant à {column}."

    if "RATING" in column:
        return f"Rating NBA correspondant à {column}."

    return f"Statistique NBA : {column}."

In [37]:
def get_unit(column):
    if column in [
        "PLAYER_ID",
        "TEAM_ID",
    ]:
        return "Identifiant"

    if column == "SEASON":
        return "Saison"

    if column == "EXP_YEARS":
        return "Saisons"

    if column == "ROSTER_AVAILABLE":
        return "Indicateur binaire"

    if column in [
        "PLAYER_NAME",
        "NICKNAME",
        "POSITION",
        "COUNTRY",
        "SCHOOL",
        "COLLEGE",
        "TEAM_ABBREVIATION",
        "AGE_GROUP",
        "EXPERIENCE_GROUP",
    ]:
        return "Catégorie"

    if column == "AGE":
        return "Années"

    if column == "EXPERIENCE":
        return "Saisons"

    if "PCT" in column or column.endswith("_RATE"):
        return "Ratio / pourcentage"

    if column.endswith("_PG"):
        return "Par match"

    if "RATING" in column:
        return "Rating"

    if column.startswith("HAS_") or column.startswith("IS_"):
        return "Indicateur binaire"

    if column == "MULTI_TEAM_SEASON":
        return "Indicateur binaire"

    return "Valeur numérique / catégorie"


def get_transformation(column):
    if column == "pts_next_season":
        return (
            "Décalage de PTS vers la saison suivante uniquement "
            "si N+1 est consécutive."
        )

    if column in derived_descriptions:
        return "Variable calculée lors du feature engineering."

    return "Nettoyage, harmonisation et jointure des sources NBA."


def get_values_or_range(series):
    non_null = series.dropna()

    if non_null.empty:
        return "Aucune valeur disponible"

    if pd.api.types.is_numeric_dtype(series):
        return (
            f"{non_null.min():.3f} à "
            f"{non_null.max():.3f}"
        )

    unique_values = (
        non_null
        .astype(str)
        .unique()
        .tolist()
    )

    if len(unique_values) <= 12:
        return ", ".join(
            sorted(unique_values)
        )

    return (
        f"{len(unique_values)} valeurs distinctes"
    )

In [38]:
dictionary_rows = []


for column in df.columns:

    dictionary_rows.append({
        "colonne":
            column,

        "type":
            str(df[column].dtype),

        "description":
            get_description(column),

        "unite":
            get_unit(column),

        "valeurs_ou_plage":
            get_values_or_range(
                df[column]
            ),

        "source":
            get_source(column),

        "transformation":
            get_transformation(column),

        "nb_manquants":
            int(
                df[column]
                .isna()
                .sum()
            ),

        "pct_manquant":
            round(
                df[column]
                .isna()
                .mean()
                * 100,
                2
            ),

        "nb_uniques":
            int(
                df[column]
                .nunique(
                    dropna=True
                )
            ),
    })


data_dictionary = pd.DataFrame(
    dictionary_rows
)


print(
    "Colonnes documentées :",
    len(data_dictionary)
)

print(
    "Colonnes du dataset :",
    df.shape[1]
)


display(
    data_dictionary.head(20)
)

Colonnes documentées : 251
Colonnes du dataset : 251


,colonne,type,description,unite,valeurs_ou_plage,source,transformation,nb_manquants,pct_manquant,nb_uniques
0,PLAYER_ID,int64,Identifiant NBA unique du joueur.,Identifiant,3.000 à 1630846.000,NBA API - statistiques joueur,"Nettoyage, harmonisation et jointure des sourc...",0,0.0,1909
1,PLAYER_NAME,string,Nom complet du joueur.,Catégorie,1906 valeurs distinctes,NBA API - statistiques joueur,"Nettoyage, harmonisation et jointure des sourc...",0,0.0,1906
2,SEASON,string,Saison NBA correspondant à l'observation.,Saison,26 valeurs distinctes,NBA API - statistiques joueur,"Nettoyage, harmonisation et jointure des sourc...",0,0.0,26
3,SEASON_START_YEAR,int64,Statistique NBA : SEASON_START_YEAR.,Valeur numérique / catégorie,1996.000 à 2021.000,NBA API - statistiques joueur,"Nettoyage, harmonisation et jointure des sourc...",0,0.0,26
4,SEASON_INDEX,int64,Statistique NBA : SEASON_INDEX.,Valeur numérique / catégorie,0.000 à 25.000,NBA API - statistiques joueur,"Nettoyage, harmonisation et jointure des sourc...",0,0.0,26
5,TEAM_ID,int64,Identifiant NBA de l'équipe.,Identifiant,1610612737.000 à 1610612766.000,NBA API - statistiques équipe,"Nettoyage, harmonisation et jointure des sourc...",0,0.0,30
6,TEAM_ABBREVIATION,object,Abréviation de l'équipe.,Catégorie,36 valeurs distinctes,NBA API - statistiques équipe,"Nettoyage, harmonisation et jointure des sourc...",0,0.0,36
7,AGE,float64,Âge du joueur lors de la saison.,Années,18.000 à 42.000,NBA API - biographie joueur,"Nettoyage, harmonisation et jointure des sourc...",0,0.0,25
8,AGE_GROUP,category,Classe d'âge du joueur.,Catégorie,"22_or_less, 23_27, 28_31, 32_plus",Feature engineering Phase 4,Variable calculée lors du feature engineering.,0,0.0,4
9,POSITION,object,Poste indiqué dans le roster NBA.,Catégorie,"C, C-F, F, F-C, F-G, G, G-F",NBA API - roster,"Nettoyage, harmonisation et jointure des sourc...",505,5.1,7


In [39]:
assert len(data_dictionary) == df.shape[1]

assert not data_dictionary[
    "description"
].isna().any()

assert not data_dictionary[
    "source"
].isna().any()

assert not data_dictionary[
    "transformation"
].isna().any()


print(
    "Data Dictionary complet."
)

Data Dictionary complet.


In [40]:
DATA_DICTIONARY_PATH = (
    DOC_DIR
    / "data_dictionary.csv"
)


data_dictionary.to_csv(
    DATA_DICTIONARY_PATH,
    index=False,
    encoding="utf-8"
)


print(
    "Data Dictionary sauvegardé :",
    DATA_DICTIONARY_PATH
)

Data Dictionary sauvegardé : c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\docs\data_dictionary.csv


## 6. Documentation des transformations

Les principales transformations appliquées depuis les données brutes jusqu'au
dataset final sont documentées afin de garantir la traçabilité du projet.

In [41]:
transformations_text = """# Transformations du dataset NBA

## 1. Sources utilisées

Le projet combine plusieurs sources NBA API :

- biographie des joueurs ;
- statistiques individuelles de base ;
- statistiques individuelles avancées ;
- statistiques collectives de base ;
- statistiques collectives avancées ;
- rosters.

Un dataset Kaggle a également été étudié lors des premières phases mais n'a
pas été joint au dataset final en raison de l'absence de PLAYER_ID et du risque
d'ambiguïté lié aux homonymes.

## 2. Nettoyage

Les principales opérations de nettoyage sont :

- suppression des colonnes techniques inutiles ;
- harmonisation des types ;
- standardisation des unités physiques ;
- conservation des valeurs manquantes lorsqu'elles possèdent un sens métier ;
- création d'indicateurs de disponibilité ;
- absence de suppression arbitraire des outliers.

## 3. Jointures

Le grain final correspond à une ligne par joueur et par saison.

Les jointures utilisent principalement :

- PLAYER_ID ;
- SEASON ;
- TEAM_ID lorsque le contexte équipe est nécessaire.

## 4. Création de la cible

La cible `pts_next_season` correspond aux points par match du joueur lors de la
saison suivante.

Elle est créée uniquement lorsque la saison suivante est consécutive.

Les lignes ne possédant pas de saison N+1 consécutive ne font pas partie du
dataset supervisé.

## 5. Feature engineering

Les principales variables créées sont :

- statistiques de la saison précédente ;
- variations par rapport à la saison précédente ;
- moyennes historiques calculées uniquement sur le passé ;
- statistiques par minute ;
- ratios offensifs ;
- groupes d'âge ;
- groupes d'expérience ;
- indicateurs rookie ;
- indicateurs multi-équipe ;
- disponibilité des données roster.

## 6. Préparation Machine Learning

Les identifiants et colonnes non généralisables sont exclus des features.

Les variables numériques sont :

- imputées par la médiane ;
- standardisées avec StandardScaler.

Les variables catégorielles sont :

- normalisées pour les valeurs manquantes ;
- imputées par la modalité la plus fréquente ;
- encodées avec OneHotEncoder.

Toutes ces transformations sont intégrées dans la Pipeline Machine Learning.

## 7. Découpage temporel

Le dataset est séparé chronologiquement :

- train : 1996-97 à 2011-12 ;
- validation : 2012-13 à 2016-17 ;
- test : 2017-18 à 2021-22.

La validation croisée respecte également l'ordre temporel.

## 8. Limites

Le dataset supervisé contient uniquement les joueurs observés lors d'une saison
N+1 consécutive.

Les retraites, sorties de ligue et saisons entièrement manquées ne sont donc pas
représentées par la cible.

Les blessures, changements de rôle, contrats et certains mouvements d'effectif
ne sont pas explicitement modélisés.
"""


TRANSFORMATIONS_PATH = (
    DOC_DIR
    / "transformations.md"
)


TRANSFORMATIONS_PATH.write_text(
    transformations_text,
    encoding="utf-8"
)


print(
    "Documentation créée :",
    TRANSFORMATIONS_PATH
)

Documentation créée : c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\docs\transformations.md


In [42]:
from importlib.metadata import (
    version,
    PackageNotFoundError,
)


packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "joblib",
    "pyarrow",
    "nba_api",
    "jupyter",
]


requirements = []


for package in packages:

    try:
        package_version = version(
            package
        )

        requirements.append(
            f"{package}=={package_version}"
        )

    except PackageNotFoundError:
        requirements.append(
            package
        )


REQUIREMENTS_PATH = (
    PROJECT_ROOT
    / "requirements.txt"
)


REQUIREMENTS_PATH.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)


print(
    REQUIREMENTS_PATH.read_text(
        encoding="utf-8"
    )
)

pandas==2.2.3
numpy==2.2.4
scikit-learn==1.7.0
matplotlib==3.10.1
joblib==1.5.1
pyarrow==21.0.0
nba_api==1.10.0
jupyter



In [43]:
changelog = f"""# Changelog

## v1.0 - {date.today().isoformat()}

- Extraction multi-sources NBA API
- Nettoyage et diagnostic des données
- Jointure des données joueur, équipe et roster
- Création de la cible pts_next_season
- Feature engineering temporel
- EDA analytique autour de cinq questions business
- Visualisations et storytelling
- Comparaison Ridge, Decision Tree et Random Forest
- Sélection du Ridge alpha=100
- Evaluation temporelle finale
- Export du dataset final
- Export de la pipeline Machine Learning
- Création du Data Dictionary
- Documentation des transformations
"""


CHANGELOG_PATH = (
    PROJECT_ROOT
    / "CHANGELOG.md"
)


CHANGELOG_PATH.write_text(
    changelog,
    encoding="utf-8"
)


print(
    "Changelog créé :",
    CHANGELOG_PATH
)

Changelog créé : c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\CHANGELOG.md


In [47]:
deliverables = {
    "Dataset Parquet": PARQUET_EXPORT,
    "Dataset CSV": CSV_EXPORT,
    "Pipeline ML": MODEL_PATH,
    "Métadonnées modèle": METADATA_PATH,
    "Data Dictionary": DATA_DICTIONARY_PATH,
    "Transformations": TRANSFORMATIONS_PATH,
    "Requirements": REQUIREMENTS_PATH,
    "Changelog": CHANGELOG_PATH,
    "README": PROJECT_ROOT / "README.md",
    "Module preprocessing": PROJECT_ROOT / "src" / "preprocessing.py",
}


delivery_check = pd.DataFrame([
    {
        "livrable": name,
        "existe": path.exists(),
        "taille_Ko": round(path.stat().st_size / 1024, 1)
        if path.exists()
        else None,
        "chemin": str(path),
    }

    for name, path in deliverables.items()
])


display(delivery_check)

,livrable,existe,taille_Ko,chemin
0,Dataset Parquet,True,2905.0,c:\Users\USER\OneDrive\LiveCampus\master\cours...
1,Dataset CSV,True,11514.3,c:\Users\USER\OneDrive\LiveCampus\master\cours...
2,Pipeline ML,True,34.5,c:\Users\USER\OneDrive\LiveCampus\master\cours...
3,Métadonnées modèle,True,0.8,c:\Users\USER\OneDrive\LiveCampus\master\cours...
4,Data Dictionary,True,47.6,c:\Users\USER\OneDrive\LiveCampus\master\cours...
5,Transformations,True,2.9,c:\Users\USER\OneDrive\LiveCampus\master\cours...
6,Requirements,True,0.1,c:\Users\USER\OneDrive\LiveCampus\master\cours...
7,Changelog,True,0.6,c:\Users\USER\OneDrive\LiveCampus\master\cours...
8,README,True,6.5,c:\Users\USER\OneDrive\LiveCampus\master\cours...
9,Module preprocessing,True,0.4,c:\Users\USER\OneDrive\LiveCampus\master\cours...


In [48]:
assert delivery_check["existe"].all()
assert df.shape == (9911, 251)
assert len(data_dictionary) == 251

print("Tous les livrables de la Phase 8 sont présents.")
print("Validation finale réussie.")

Tous les livrables de la Phase 8 sont présents.
Validation finale réussie.


## Questions de réflexion

### Reproductibilité

Le projet contient le dataset final, la pipeline Machine Learning, les
dépendances, le Data Dictionary et la documentation des transformations.

Les principaux éléments nécessaires pour comprendre et reproduire l'analyse
sont donc disponibles.

### Pérennité

La structure des dossiers, le versionnement et la documentation permettent de
reprendre le projet ultérieurement sans dépendre uniquement des notebooks.

### Partage

Le dataset est disponible en Parquet pour les usages techniques et en CSV pour
faciliter son utilisation avec d'autres outils.

### Amélioration

Une future version pourrait automatiser davantage le workflow et utiliser DVC
pour versionner les fichiers de données volumineux.

## Conclusion de la Phase 8

Le dataset final a été exporté en Parquet et CSV et sa relecture a été vérifiée.

La pipeline Ridge finale est sauvegardée avec son preprocessing complet et peut
être rechargée directement pour effectuer de nouvelles prédictions.

Le projet possède également un Data Dictionary couvrant les 251 colonnes, les
métadonnées du modèle, la documentation des transformations, les dépendances,
un changelog et un README.

Les principaux livrables nécessaires à la reproductibilité et au partage sont
donc disponibles.

La prochaine étape est la Phase 9 : Présentation et Soutenance.